# Convolution: filters that slide
**Machine Learning · Week 9 (Deep learning: images, sequences and attention) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w9d1.html

A **convolution** slides a small grid of weights, a **filter** (or kernel), over an image. At every position
it multiplies the filter with the patch underneath and adds everything up: one number per position, a new
image called a **feature map**. Here it is by hand, on a photo that comes with scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

photo = load_sample_image("china.jpg").mean(axis=2) / 255     # (427, 640), gray, values 0 to 1
photo = photo[::2, ::2]                                          # half size: quicker by hand

def convolve(image, kernel):
    """Slide the kernel over the image (no padding, stride 1): multiply and add at every position."""
    k = kernel.shape[0]
    H, W = image.shape
    out = np.zeros((H - k + 1, W - k + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i + k, j:j + k] * kernel)
    return out

vertical_edges = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])                          # the Sobel filter: right side minus left side
edges = convolve(photo, vertical_edges)
print("image", photo.shape, "-> feature map", edges.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
ax1.imshow(photo, cmap="gray")
ax1.set_title("The photo")
ax2.imshow(edges, cmap="RdBu_r", vmin=-2, vmax=2)
ax2.set_title("After the vertical-edge filter")
for ax in (ax1, ax2):
    ax.axis("off")
plt.show()

## Different filters, different features
The same sliding, with different weights, finds different things: blur averages a patch, sharpen exaggerates
a pixel against its neighbors, and edge filters respond where brightness changes in one direction.

In [ ]:
filters = {
    "blur": np.ones((3, 3)) / 9,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "vertical edges": vertical_edges,
    "horizontal edges": vertical_edges.T,
}
crop = photo[40:160, 60:220]
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), layout="constrained")
for ax, (name, kernel) in zip(axes, filters.items()):
    out = convolve(crop, kernel)
    edge = "edges" in name
    ax.imshow(out, cmap="RdBu_r" if edge else "gray", vmin=-2 if edge else 0, vmax=2 if edge else 1)
    ax.set_title(name)
    ax.axis("off")
plt.show()

## The same thing in PyTorch, much faster
`torch.nn.functional.conv2d` expects a batch of images with channels, `(batch, channels, height, width)`,
and a stack of filters, `(filters, channels, k, k)`. Strictly it computes a *cross-correlation* (the filter
isn't flipped), which is what deep learning calls convolution.

In [ ]:
import time
import torch
import torch.nn.functional as F

img = torch.tensor(photo, dtype=torch.float32)[None, None]           # (1, 1, H, W)
bank = torch.tensor(np.stack(list(filters.values())), dtype=torch.float32)[:, None]   # (4, 1, 3, 3)
t0 = time.perf_counter()
maps = F.conv2d(img, bank)
print(f"PyTorch: 4 filters in {1000 * (time.perf_counter() - t0):.1f} ms, output shape {tuple(maps.shape)}")
print("same as by hand:", np.allclose(maps[0, 2].numpy(), edges, atol=1e-4))

## Padding, stride and pooling: the sizes
A $k \times k$ filter on an $n \times n$ image gives $(n - k + 2p)/s + 1$ outputs per side, with padding $p$
(zeros added around the border) and stride $s$ (how far the filter jumps). **Max pooling** keeps the largest
value in each $2 \times 2$ block: half the size, and the strongest responses survive.

In [ ]:
x = torch.rand(1, 1, 28, 28)
for name, layer in [("3x3, no padding", torch.nn.Conv2d(1, 1, 3)),
                    ("3x3, padding 1", torch.nn.Conv2d(1, 1, 3, padding=1)),
                    ("3x3, padding 1, stride 2", torch.nn.Conv2d(1, 1, 3, padding=1, stride=2)),
                    ("2x2 max pooling", torch.nn.MaxPool2d(2))]:
    print(f"{name:26s} 28 x 28 -> {tuple(layer(x).shape[2:])}")

## Your turn: a diagonal edge detector
Design a 3 x 3 filter that responds to edges running from bottom-left to top-right (a "/" shape), apply it to
`crop` with `convolve`, and show the result next to the vertical-edge map. Hint: the weights should add up to 0,
so flat areas give 0.

In [ ]:
diagonal = np.array([[0, 0, 0],
                     [0, 0, 0],
                     [0, 0, 0]])          # TODO: your weights
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3), layout="constrained")
ax1.imshow(convolve(crop, vertical_edges), cmap="RdBu_r", vmin=-2, vmax=2)
ax1.set_title("vertical edges")
# TODO: show convolve(crop, diagonal) in ax2
plt.show()

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
diagonal = np.array([[-2, -1, 0],
                     [-1, 0, 1],
                     [0, 1, 2]])          # bottom-right side minus top-left side: zero along "/" lines, sum 0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3), layout="constrained")
ax1.imshow(convolve(crop, vertical_edges), cmap="RdBu_r", vmin=-2, vmax=2)
ax1.set_title("vertical edges")
ax2.imshow(convolve(crop, diagonal), cmap="RdBu_r", vmin=-2, vmax=2)
ax2.set_title("diagonal (/) edges")
for ax in (ax1, ax2):
    ax.axis("off")
plt.show()
print("The filter compares the two sides of a '/' line, so it lights up wherever brightness changes across one.")